In [11]:
%pip install -U sentence-transformers datasets accelerate

  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 20.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 855.2/855.2 kB 61.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 62.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 118.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 59.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 28.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.9/548.9 kB 34.0 MB/s  0:00:00
Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl (484 kB)
Using cached rich-15.0.0-py3-none-any.whl (310 kB)
Using cached markdown_it_py-4.2.0-py3-none-any.whl (91 kB)
Us

In [13]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd

restaurant = pd.read_parquet("restaurant_global.parquet")
review = pd.read_parquet("review_global.parquet")

print("Restaurants:", restaurant.shape)
print("Reviews:", review.shape)

restaurant_info = restaurant[["business_id", "name", "categories", "city", "stars"]].copy()

df = review.merge(restaurant_info,on="business_id",how="inner",validate="many_to_one")

df.drop()
df.head()

# join the two parquets. 

Restaurants: (52286, 9)
Reviews: (4724684, 9)


,review_id,user_id,business_id,stars_x,useful,funny,cool,text,date,name,categories,city,stars_y
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11,Turning Point of North Wales,"Restaurants, Breakfast & Brunch, Food, Juice B...",North Wales,3.0
1,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30,Kettle Restaurant,"Restaurants, Breakfast & Brunch",Tucson,3.5
2,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03,Zaika,"Halal, Pakistani, Restaurants, Indian",Philadelphia,4.0
3,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15,Melt,"Sandwiches, Beer, Wine & Spirits, Bars, Food, ...",New Orleans,4.0
4,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1,1,2,1,I am a long term frequent customer of this est...,2015-09-23 23:10:31,Dmitri's,"Mediterranean, Restaurants, Seafood, Greek",Philadelphia,4.0


In [2]:
df["categories"]

0          Restaurants, Breakfast & Brunch, Food, Juice B...
1                            Restaurants, Breakfast & Brunch
2                      Halal, Pakistani, Restaurants, Indian
3          Sandwiches, Beer, Wine & Spirits, Bars, Food, ...
4                 Mediterranean, Restaurants, Seafood, Greek
                                 ...                        
4724679                Restaurants, Burgers, Food, Fast Food
4724680    Bars, Restaurants, American (New), Beer Bar, N...
4724681    Halal, Lebanese, Mediterranean, Greek, Restaur...
4724682    Desserts, Pizza, Salad, Food, Beer, Wine & Spi...
4724683    Burgers, Sandwiches, Food, Beer, Wine & Spirit...
Name: categories, Length: 4724684, dtype: str

In [4]:
training_df = df.sample(n=min(100_000, len(df)),random_state=42).copy()

training_df["anchor"] = ("Restaurant categories: " + training_df["categories"])
training_df["positive"] = training_df["text"]
training_df = training_df[["business_id", "anchor", "positive"]].drop_duplicates(subset=["anchor", "positive"])

print("Training pairs:", len(training_df))

train_dataset = training_df.reset_index(drop=True)

print(train_dataset.head())

Training pairs: 99994
              business_id                                             anchor  \
0  GBD-YjcOD1nG0wv4L0Lvfg  Restaurant categories: American (Traditional),...   
1  gVU0U7gNBMQxKTzqmRtdLg  Restaurant categories: Burgers, Steakhouses, A...   
2  4RLS-xn9hHxm8-HTLXf8Og        Restaurant categories: Mexican, Restaurants   
3  mhrW9O0O5hXGXGnEYBVoag  Restaurant categories: Seafood, Cajun/Creole, ...   
4  X7yGrz7oDbt8UY5rdmJc_w  Restaurant categories: Pakistani, Himalayan/Ne...   

                                            positive  
0  I'm not a religious woman but I do think this ...  
1  Miguel was a fabulous bartender!!!  Super nice...  
2  I had the chicken mole.   It was excellent.   ...  
3  Waited next door at the Maple Leaf for a table...  
4  Given the space and limited number of small ta...  


In [14]:
from datasets import Dataset
pair_ls = [(pos, anch) for pos, anch in zip(training_df["positive"], training_df["anchor"])]

trn = pair_ls[:80000]
tst = pair_ls[80000:]

train_dataset = Dataset.from_dict({"anchor": [pos for pos, anch in trn],"positive": [anch for pos, anch in trn]})

test_dataset = Dataset.from_dict({"anchor": [pos for pos, anch in tst],"positive": [anch for pos, anch in tst]})

In [15]:
len(tst)

19994

In [40]:
from sentence_transformers import (SentenceTransformer,SentenceTransformerTrainer,SentenceTransformerTrainingArguments)
from sentence_transformers.losses import MultipleNegativesRankingLoss


model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir="restaurant-model-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=3e-04,
    warmup_ratio=0.1,
    save_strategy="epoch",
    logging_steps=100,
    report_to="none"
)

trainer = SentenceTransformerTrainer(model=model,args=args,train_dataset=train_dataset,loss=loss)

trainer.train()

model.save_pretrained("restaurant-model-finetuned")

print("Fine-tuning complete!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/opt/anaconda3/envs/csci5800/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
100,1.974098
200,1.717267
300,1.646128
400,1.587224
500,1.596253
600,1.542371
700,1.574670
800,1.554639
900,1.551942
1000,1.530036


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/envs/csci5800/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete!


In [41]:
from sentence_transformers import util

model = SentenceTransformer("restaurant-model-finetuned")

categories = "Restaurants, Food, Bubble Tea, Coffee & Tea, Bakeries"
review = "The bubble tea was delicious and the pastries were fresh."

embeddings = model.encode([categories, review],convert_to_tensor=True,normalize_embeddings=True)

similarity = util.cos_sim(embeddings[0], embeddings[1]).item()

print(f"Cosine similarity: {similarity:.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Cosine similarity: 0.9016


In [42]:
import numpy as np
import torch
from sentence_transformers import util

anchors = test_dataset["anchor"]
positives = test_dataset["positive"]

n = min(1000, len(anchors))
anchors = anchors[:n]
positives = positives[:n]

anchor_embeddings = model.encode(
    anchors,
    batch_size=64,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

positive_embeddings = model.encode(
    positives,
    batch_size=64,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

scores = util.cos_sim(anchor_embeddings, positive_embeddings)

ranked_indices = scores.argsort(dim=1, descending=True)

# Fix: create target indices on the same device
targets = torch.arange(n, device=scores.device).unsqueeze(1)

correct_positions = (
    ranked_indices == targets
).nonzero(as_tuple=True)[1]

top1 = (correct_positions == 0).float().mean().item()
top5 = (correct_positions < min(5, n)).float().mean().item()
top10 = (correct_positions < min(10, n)).float().mean().item()

print(f"Test examples: {n}")
print(f"Top-1 accuracy: {top1:.2%}")
print(f"Recall@5:       {top5:.2%}")
print(f"Recall@10:      {top10:.2%}")

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Test examples: 1000
Top-1 accuracy: 9.50%
Recall@5:       26.60%
Recall@10:      38.40%
